# EXECUTION CODE

In [19]:
import cloudpickle
import pandas as pd
import numpy as np
from janitor import clean_names

from kkbox_churn.config import PROJECT_ROOT, SELECTED_FEATURES

# We are using the validation file here, but in production this should be replaced by the new data to score
validation_path = PROJECT_ROOT / "02_data"/ "validation" / "validation.csv"
df = pd.read_csv(validation_path, index_col=0)

# Selection, cleaning and transformation
df = (
    clean_names(df)       
    .rename(columns = {'bd':'age'})  
)

X_new = df[SELECTED_FEATURES].copy()

# Load execution pipeline
execution_pipeline_path = PROJECT_ROOT / "04_models" / "execution_pipeline.pickle"

with open(execution_pipeline_path, mode="rb") as file:
    execution_pipeline = cloudpickle.load(file)

# Get scores (probability of the positive class)
scoring = execution_pipeline.predict_proba(X_new)[:, 1]

In [24]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Etiquetas reales
y_true = df["is_churn"].copy()

# Probabilidades 
scoring = execution_pipeline.predict_proba(X_new)[:, 1]

# Umbral seleccionado previamente en validación interna
threshold = 0.08606863975870534

# Clasificación binaria
y_pred = (scoring >= threshold).astype(int)

# Métricas
metrics = {
    "roc_auc": roc_auc_score(y_true, scoring),
    "average_precision": average_precision_score(y_true, scoring),
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1": f1_score(y_true, y_pred, zero_division=0)
}

print("RESULTADOS EN EL HOLDOUT FINAL")
display(pd.Series(metrics, name="Valor").round(4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification report:")
print(classification_report(y_true, y_pred, digits=3))

print(f"\nClientes clasificados como churn: {y_pred.mean():.2%}")
print(f"Churn real: {y_true.mean():.2%}")


RESULTADOS EN EL HOLDOUT FINAL


roc_auc              0.8544
average_precision    0.6337
accuracy             0.6228
precision            0.1755
recall               0.8635
f1                   0.2917
Name: Valor, dtype: float64


Matriz de confusión:
[[158787 106302]
 [  3576  22623]]

Classification report:
              precision    recall  f1-score   support

           0      0.978     0.599     0.743    265089
           1      0.175     0.864     0.292     26199

    accuracy                          0.623    291288
   macro avg      0.577     0.731     0.517    291288
weighted avg      0.906     0.623     0.702    291288


Clientes clasificados como churn: 44.26%
Churn real: 8.99%
